# 🧠 El Ciclo Básico de Deep Learning en Lenguaje (NLP) con PyTorch Puro (Sin Arnés)

> **Objetivo:** Ver y comprender todo el ciclo de entrenamiento de una red neuronal para **procesamiento de lenguaje natural (NLP)** paso a paso, usando **exclusivamente PyTorch estándar** (sin librerías intermedias ni el arnés `harness.py`).

---

### ¿Cómo fluye el texto en una red neuronal?

```none
┌───────────────┐     ┌──────────────┐     ┌──────────────────┐     ┌──────────────────┐
│ Frase (Texto) │ ──> │ IDs (Tokens) │ ──> │ Vectores Densos  │ ──> │ Clasificación    │
│ "me encanta"  │     │  [ 4, 12 ]   │     │ (nn.Embedding)   │     │ (Logits / Clases)│
└───────────────┘     └──────────────┘     └──────────────────┘     └──────────────────┘
```

### Las 4 fases fundamentales del bucle de entrenamiento:

```none
┌──────────────────────────────────────────────────────────┐
│ 1. PREDECIR   la red produce una salida       (forward)  │
│ 2. MEDIR      comparar con lo esperado        (pérdida)  │
│ 3. CULPAR     ¿cuánto contribuyó cada peso?   (backward) │
│ 4. CORREGIR   ajustar cada peso un poco       (update)   │
└──────────────────────────────────────────────────────────┘
```

## 0. Importaciones y configuración del entorno

In [ ]:
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

# 1. Semillas para asegurar reproducibilidad exacta
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# 2. Selección de dispositivo (GPU si está disponible, sino CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando dispositivo: {device}")

## 1. Datos de Texto, Vocabulario y `DataLoader`

Creamos un conjunto de datos sencillo para **Clasificación de Sentimientos**:
- Clase `0`: Sentimiento Negativo / Crítica insatisfecha.
- Clase `1`: Sentimiento Positivo / Crítica satisfecha.

In [ ]:
# 1. Corpus de texto sintético etiquetado
raw_train_data = [
    ("este producto es excelente y de gran calidad", 1),
    ("me encanta funciona perfecto y es muy rapido", 1),
    ("una maravilla totalmente recomendado para todos", 1),
    ("muy contento con la compra super util y bonito", 1),
    ("la atencion al cliente fue fantastica y rapida", 1),
    ("espectacular llego a tiempo y en perfecto estado", 1),
    ("un diseno maravilloso y una calidad sobresaliente", 1),
    ("funciona increible muy satisfecho con el resultado", 1),
    
    ("es una perdida total de dinero y tiempo", 0),
    ("horrible experiencia el producto llego roto", 0),
    ("muy lento mala calidad y no funciona", 0),
    ("pesimo servicio no lo recomiendo para nada", 0),
    ("no sirve para nada estoy muy decepcionado", 0),
    ("muy mala atencion y producto defectuoso", 0),
    ("terrible calidad se rompio al primer dia", 0),
    ("una completa estafa no compren este articulo", 0),
]

raw_val_data = [
    ("muy buen producto y atencion excelente", 1),
    ("llego rapido y funciona de maravilla", 1),
    ("pesima compra muy decepcionado y roto", 0),
    ("no funciona nada una perdida de dinero", 0),
]

# 2. Construcción del Vocabulario (mapeo palabra -> id numérico)
PAD_TOKEN = "<pad>"  # Token especial de relleno (índice 0)
UNK_TOKEN = "<unk>"  # Token para palabras no vistas (índice 1)

vocab = {PAD_TOKEN: 0, UNK_TOKEN: 1}
for text, _ in raw_train_data:
    for word in text.lower().split():
        if word not in vocab:
            vocab[word] = len(vocab)

print(f"Vocabulario construido: {len(vocab)} palabras únicas")

# 3. Función auxiliar para transformar texto en tensor de IDs
MAX_LEN = 8  # Longitud fija para las secuencias

def text_to_tensor(text, vocab, max_len=MAX_LEN):
    tokens = text.lower().split()
    ids = [vocab.get(w, vocab[UNK_TOKEN]) for w in tokens]
    # Rellenar (padding) o truncar a longitud fija
    if len(ids) < max_len:
        ids += [vocab[PAD_TOKEN]] * (max_len - len(ids))
    else:
        ids = ids[:max_len]
    return torch.tensor(ids, dtype=torch.long)

# 4. Clase Dataset personalizada para texto
class TextSentimentDataset(Dataset):
    def __init__(self, data, vocab, max_len=MAX_LEN):
        self.samples = []
        for text, label in data:
            x = text_to_tensor(text, vocab, max_len)
            y = torch.tensor(label, dtype=torch.long)
            self.samples.append((x, y))
            
    def __len__(self):
        return len(self.samples)
        
    def __getitem__(self, idx):
        return self.samples[idx]

train_dataset = TextSentimentDataset(raw_train_data, vocab)
val_dataset = TextSentimentDataset(raw_val_data, vocab)

BATCH_SIZE = 4
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Muestras de entrenamiento: {len(train_dataset)} ({len(train_loader)} lotes de {BATCH_SIZE})")
print(f"Muestras de validación:    {len(val_dataset)} ({len(val_loader)} lotes de {BATCH_SIZE})")

## 2. Definición del Modelo de Lenguaje (`nn.Module`)

Construimos una arquitectura de clasificación de texto con:
1. **`nn.Embedding`:** Proyecta cada palabra (ID entero) en un vector denso continuo de dimensión `embed_dim`.
2. **Global Average Pooling:** Promedia los vectores de todas las palabras de la frase para condensarla en una representación fija.
3. **Clasificador Lineal:** Capas `nn.Linear` + activación `ReLU` para predecir las probabilidades de clase (Negativo / Positivo).

In [ ]:
class TextSentimentClassifier(nn.Module):
    def __init__(self, vocab_size: int, embed_dim: int = 16, hidden_dim: int = 16, num_classes: int = 2):
        super().__init__()
        # 1. Capa de Embedding (padding_idx=0 mantiene el vector de <pad> en ceros)
        self.embedding = nn.Embedding(num_embeddings=vocab_size, embedding_dim=embed_dim, padding_idx=0)
        
        # 2. Red densa clasificadora
        self.classifier = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, num_classes)  # Salida: Logits para [Clase 0, Clase 1]
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Entrada x: [batch_size, seq_len]
        # 1. Obtener embeddings de las palabras -> [batch_size, seq_len, embed_dim]
        embeds = self.embedding(x)
        
        # 2. Promediar a lo largo de la secuencia (dimensión 1) -> [batch_size, embed_dim]
        pooled = embeds.mean(dim=1)
        
        # 3. Pasar por el clasificador -> [batch_size, num_classes]
        logits = self.classifier(pooled)
        return logits

# Instanciar el modelo y moverlo a la memoria adecuada (CPU o GPU)
model = TextSentimentClassifier(vocab_size=len(vocab), embed_dim=16, hidden_dim=16, num_classes=2).to(device)
print(model)

## 3. Función de Pérdida y Optimizador

- **Función de pérdida (Loss):** Entropía Cruzada (`CrossEntropyLoss`), estándar para problemas de clasificación multiclase.
- **Optimizador:** `Adam` con tasa de aprendizaje (Learning Rate) $\eta = 0.01$ sobre los parámetros `model.parameters()`.

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-2)
epochs = 35

## 4. El Bucle de Entrenamiento (Secuencial en PyTorch)

En cada época:
1. **Entrenamiento (`model.train()`):** Se itera por los lotes de texto y se aplican las 4 fases (`forward`, `loss`, `backward`, `optimizer.step()`).
2. **Validación (`model.eval()` con `torch.no_grad()`):** Se evalúa la pérdida y el acierto (*accuracy*) sobre frases no vistas.

In [ ]:
history = []  # Para registrar las métricas de cada época

for epoch in range(epochs):
    # =========================================================================
    # FASE A: ENTRENAMIENTO
    # =========================================================================
    model.train()
    total_train_loss = 0.0
    correct_train = 0
    total_train_samples = 0

    for inputs, targets in train_loader:
        inputs, targets = inputs.to(device), targets.to(device)

        # --- PASO 0: Limpiar gradientes anteriores ---
        optimizer.zero_grad()

        # --- PASO 1: PREDECIR (Forward) ---
        logits = model(inputs)

        # --- PASO 2: MEDIR (Pérdida / Loss) ---
        loss = criterion(logits, targets)

        # --- PASO 3: CULPAR (Backward / Gradientes) ---
        loss.backward()

        # --- PASO 4: CORREGIR (Update de pesos) ---
        optimizer.step()

        # Métricas de entrenamiento
        total_train_loss += loss.item() * len(inputs)
        preds = logits.argmax(dim=1)
        correct_train += (preds == targets).sum().item()
        total_train_samples += len(inputs)

    mean_train_loss = total_train_loss / total_train_samples
    train_accuracy = correct_train / total_train_samples

    # =========================================================================
    # FASE B: EVALUACIÓN / VALIDACIÓN
    # =========================================================================
    model.eval()
    total_val_loss = 0.0
    correct_val = 0
    total_val_samples = 0

    with torch.no_grad():
        for val_inputs, val_targets in val_loader:
            val_inputs, val_targets = val_inputs.to(device), val_targets.to(device)

            val_logits = model(val_inputs)
            val_loss = criterion(val_logits, val_targets)

            total_val_loss += val_loss.item() * len(val_inputs)
            val_preds = val_logits.argmax(dim=1)
            correct_val += (val_preds == val_targets).sum().item()
            total_val_samples += len(val_inputs)

    mean_val_loss = total_val_loss / total_val_samples
    val_accuracy = correct_val / total_val_samples

    # =========================================================================
    # REGISTRO Y SALIDA POR CONSOLA
    # =========================================================================
    history.append({
        "epoch": epoch,
        "train_loss": mean_train_loss,
        "val_loss": mean_val_loss,
        "train_acc": train_accuracy,
        "val_acc": val_accuracy
    })

    # Mostrar progreso cada 5 épocas y en la última
    if epoch % 5 == 0 or epoch == epochs - 1:
        print(f"  epoch {epoch:3d}  train_loss {mean_train_loss:.4f} (acc {train_accuracy*100:5.1f}%)  |  val_loss {mean_val_loss:.4f} (acc {val_accuracy*100:5.1f}%)")

## 5. Visualización de Curvas de Aprendizaje

Graficamos:
1. **Curva de Pérdida:** Reducción del error a lo largo de las épocas.
2. **Curva de Precisión (Accuracy):** Evolución del porcentaje de aciertos.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))

epochs_range = [h["epoch"] for h in history]
train_losses = [h["train_loss"] for h in history]
val_losses = [h["val_loss"] for h in history]
train_accs = [h["train_acc"] * 100 for h in history]
val_accs = [h["val_acc"] * 100 for h in history]

# 1. Curva de Pérdida (Loss)
ax1.plot(epochs_range, train_losses, label="Train Loss", color="tab:blue", lw=2)
ax1.plot(epochs_range, val_losses, label="Val Loss", color="tab:orange", linestyle="--", lw=2)
ax1.set_xlabel("Época")
ax1.set_ylabel("Pérdida (Cross-Entropy)")
ax1.set_title("Evolución de la Pérdida")
ax1.legend()
ax1.grid(alpha=0.3)

# 2. Curva de Acierto (Accuracy)
ax2.plot(epochs_range, train_accs, label="Train Accuracy", color="tab:green", lw=2)
ax2.plot(epochs_range, val_accs, label="Val Accuracy", color="tab:red", linestyle="--", lw=2)
ax2.set_xlabel("Época")
ax2.set_ylabel("Precisión (%)")
ax2.set_title("Evolución del Acierto")
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Inferencia con Frases Nuevas (Test Interactivo)

Probamos el modelo entrenado con oraciones completamente nuevas para evaluar cómo generaliza.

In [ ]:
def predict_sentiment(text: str, model: nn.Module, vocab: dict) -> tuple[str, float]:
    model.eval()
    with torch.no_grad():
        # Transformar texto en tensor y añadir dimensión de batch [1, seq_len]
        x_tensor = text_to_tensor(text, vocab).unsqueeze(0).to(device)
        logits = model(x_tensor)
        # Convertir logits a probabilidades con Softmax
        probabilities = torch.softmax(logits, dim=1)[0]
        pred_idx = probabilities.argmax().item()
        
        sentiment = "POSITIVO 😊" if pred_idx == 1 else "NEGATIVO 😞"
        confidence = probabilities[pred_idx].item() * 100
        return sentiment, confidence

test_phrases = [
    "un producto fantastico y muy util",
    "pesimo servicio muy lento y roto",
    "la atencion fue excelente y rapida",
    "no funciona para nada fue una perdida de dinero",
    "calidad maravillosa totalmente recomendado"
]

print("=== PREDICCIONES EN TEXTO NUEVO ===\n")
for phrase in test_phrases:
    sent, conf = predict_sentiment(phrase, model, vocab)
    print(f"• Frase: \"{phrase}\"")
    print(f"  → Predicción: {sent} (Confianza: {conf:.1f}%)\n")

## 7. Persistencia (Guardar los pesos)

Guardamos los parámetros entrenados del modelo en disco usando `torch.save` y su `state_dict()`:

In [ ]:
torch.save(model.state_dict(), "modelo_nlp_sentimiento.pt")
print("✅ Pesos del modelo guardados correctamente en 'modelo_nlp_sentimiento.pt'")
print(f"Métricas finales: Val Loss = {history[-1]['val_loss']:.4f} | Val Accuracy = {history[-1]['val_acc']*100:.1f}%")

---

## 💡 Comparación: Datos Numéricos vs. Lenguaje en PyTorch

| Concepto | Regresión Numérica / Sintética (Notebook `00`) | Lenguaje / NLP (Este Notebook) |
|---|---|---|
| **Forma de la Entrada ($X$)** | Tensores continuos `float32` de tamaño `[batch, 1]` | Tensores discretos de enteros `int64` (IDs) de tamaño `[batch, seq_len]` |
| **Representación** | Valores reales directos a capas lineales | Mapeo discreto mediante `nn.Embedding` a vectores densos continuos |
| **Agregación temporal** | No requerida (cada muestra es un escalar) | Global Average Pooling o RNN/Transformers sobre la dimensión temporal |
| **Función de Pérdida** | `nn.MSELoss` (distancia numérica euclídea) | `nn.CrossEntropyLoss` (probabilidades categóricas) |
| **Salida de la Red** | Un número continuo predicho $\hat{y}$ | Logits de clase clasificados vía `argmax` o `softmax` |